In [2]:
from pyspark.sql import SparkSession

In [3]:
app = (
    SparkSession.builder
        .appName("Spark Dockerizado")
        .master("local[*]")
        .config("spark.sql.shuffle.partitions", "8")
        .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/25 18:34:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
print(app.version)

3.5.1


In [5]:
print(app.sparkContext.defaultParallelism)

20


In [6]:
print(app.conf.get('spark.sql.shuffle.partitions'))

8


In [7]:
from pathlib import Path
import requests

directorio = Path("data")
directorio.mkdir(exist_ok=True)

ubicacion = directorio / "yellow_trip_data_202401.parquet"


In [8]:
# def load_data(url):
#     directorio = Path('data')

#     directorio.mkdir(exist_ok=True)
    
#     ubicacion = dirrectorio/"yellow_trip_data_202401.parquet"

#     r = requests.get(url, stream=True, timeout=120)

#     r.raise_for_status()

#     with open(ubicacion, 'wb') as archivo:
#         for chunk in r.iter_content(chunk_size=1024*1024):
#             if chunk:
#                 archivo.write(chuk)
    
    

In [9]:
from pathlib import Path
import requests

def load_data(url: str,ubicacion) :
    try:
        print("Iniciando descarga...")

       
        print(f"Guardando en: {ubicacion.resolve()}")

        r = requests.get(url, stream=True, timeout=120)
        r.raise_for_status()

        total_descargado = 0

        with open(ubicacion, "wb") as archivo:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    archivo.write(chunk)
                    total_descargado += len(chunk)
                    print(f"Descargado: {total_descargado / (1024*1024):.2f} MB")

        print("Descarga completada correctamente.")

    except requests.exceptions.RequestException as e:
        print(f"Error en la descarga: {e}")
        raise
    except Exception as e:
        print(f"Error inesperado: {e}")
        raise

In [10]:
load_data('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet', ubicacion)

Iniciando descarga...
Guardando en: /workspace/notebooks/data/yellow_trip_data_202401.parquet
Descargado: 1.00 MB
Descargado: 2.00 MB
Descargado: 3.00 MB
Descargado: 4.00 MB
Descargado: 5.00 MB
Descargado: 6.00 MB
Descargado: 7.00 MB
Descargado: 8.00 MB
Descargado: 9.00 MB
Descargado: 10.00 MB
Descargado: 11.00 MB
Descargado: 12.00 MB
Descargado: 13.00 MB
Descargado: 14.00 MB
Descargado: 15.00 MB
Descargado: 16.00 MB
Descargado: 17.00 MB
Descargado: 18.00 MB
Descargado: 19.00 MB
Descargado: 20.00 MB
Descargado: 21.00 MB
Descargado: 22.00 MB
Descargado: 23.00 MB
Descargado: 24.00 MB
Descargado: 25.00 MB
Descargado: 26.00 MB
Descargado: 27.00 MB
Descargado: 28.00 MB
Descargado: 29.00 MB
Descargado: 30.00 MB
Descargado: 31.00 MB
Descargado: 32.00 MB
Descargado: 33.00 MB
Descargado: 34.00 MB
Descargado: 35.00 MB
Descargado: 36.00 MB
Descargado: 37.00 MB
Descargado: 38.00 MB
Descargado: 39.00 MB
Descargado: 40.00 MB
Descargado: 41.00 MB
Descargado: 42.00 MB
Descargado: 43.00 MB
Descargado: 

In [11]:
df = app.read.parquet(str(ubicacion))
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [12]:
df.count()
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee']

In [13]:
df.select('VendorId').show()

# todo antes del show es lazy solo show es eager
df.select(['tip_amount','fare_amount']).describe().show()

+--------+
|VendorId|
+--------+
|       2|
|       1|
|       1|
|       1|
|       1|
|       1|
|       2|
|       1|
|       2|
|       2|
|       2|
|       1|
|       1|
|       1|
|       1|
|       1|
|       1|
|       2|
|       2|
|       2|
+--------+
only showing top 20 rows

+-------+------------------+------------------+
|summary|        tip_amount|       fare_amount|
+-------+------------------+------------------+
|  count|           2964624|           2964624|
|   mean|3.3358700158940002|18.175061916791037|
| stddev| 3.896550599806763| 18.94954770590526|
|    min|             -80.0|            -899.0|
|    max|             428.0|            5000.0|
+-------+------------------+------------------+



In [14]:
from pyspark.sql import functions as F
# viajes por dia
df_por_dia = (
    df
    .withColumn("fecha", F.to_date("tpep_pickup_datetime")) # crea una nueva columna
    .groupBy("fecha")
    .agg(F.count("*").alias("viajes"))
    .orderBy("fecha")
)

df_por_dia.show()

# esto s epued ever en el dashboard de spark en localhost:4040

[Stage 9:===========================================>              (9 + 3) / 12]

+----------+------+
|     fecha|viajes|
+----------+------+
|2002-12-31|     2|
|2009-01-01|     3|
|2023-12-31|    10|
|2024-01-01| 81013|
|2024-01-02| 75519|
|2024-01-03| 82427|
|2024-01-04|102901|
|2024-01-05|103178|
|2024-01-06| 97117|
|2024-01-07| 67543|
|2024-01-08| 80034|
|2024-01-09| 93962|
|2024-01-10| 95000|
|2024-01-11|105010|
|2024-01-12|103655|
|2024-01-13|104758|
|2024-01-14| 94420|
|2024-01-15| 77033|
|2024-01-16| 93057|
|2024-01-17|110365|
+----------+------+
only showing top 20 rows



In [15]:
# Transformacion Silver:
# trip_distance >=0
# total_amounts >=0
# pickup_ts <=
# renombrar tpep_pickup a pickup_ts y dropoff_ts

silver_df = (
    df
    # Renombrar columnas
    .withColumnRenamed("tpep_pickup_datetime", "pickup_ts")
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_ts")
    .withColumn('pickup_date', F.to_date('pickup_ts'))
    .withColumn('pickup_month', F.date_format('pickup_ts', 'yyyy-MM'))
    .withColumn('pickup_hour', F.hour('pickup_ts'))
    .withColumn(
        'trip_duration_min',
        (F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))/60.0
    )
        
    
    # Filtros de calidad
    .filter(F.col("trip_distance") >= 0)
    .filter(F.col("total_amount") >= 0)
    .filter(F.col("trip_duration_min") >= 0)
)

silver_df.printSchema()
silver_df.show(5)

root
 |-- VendorID: integer (nullable = true)
 |-- pickup_ts: timestamp_ntz (nullable = true)
 |-- dropoff_ts: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_month: string (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- trip_duration_

26/02/25 18:35:04 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+------------+-----------+------------------+
|VendorID|          pickup_ts|         dropoff_ts|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|pickup_date|pickup_month|pickup_hour| trip_duration_min|
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+------------+-----------+------------------+
|       2|2024-01-01 00:57:5

In [16]:
df.count()-silver_df.count()

35560

In [17]:
zones_directorio = directorio / 'taxi_zone_lookup.csv'
print(zones_directorio)

data/taxi_zone_lookup.csv


In [19]:
zones_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
load_data(zones_url, zones_directorio)

Iniciando descarga...
Guardando en: /workspace/notebooks/data/taxi_zone_lookup.csv
Descargado: 0.01 MB
Descarga completada correctamente.


In [21]:
df_zones = app.read.csv(str(zones_directorio), header=True, inferSchema=True) # COMO scsv no es paruqte no tiene le esquem

In [22]:
df_zones.show(5)
df_zones.printSchema()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [25]:
# Seleccionar todo dim_zones creandolo 

dim_zones_PU = df_zones.select(
    F.col('LocationID').alias('PU_location_id'),
    F.col('Borough').alias('PU_borough'),
    F.col('Zone').alias('PU_zone')
)

dim_zones_DO = df_zones.select(
    F.col('LocationID').alias('DO_location_id'),
    F.col('Borough').alias('DO_borough'),
    F.col('Zone').alias('DO_zone')
)

df_silver_enriched = (
    silver_df
    .join(dim_zones_PU, silver_df['PULocationID'] == dim_zones_PU['PU_location_id'], 'left')
    .join(dim_zones_DO, silver_df['DOLocationID'] == dim_zones_DO['DO_location_id'], 'left')
    
)

In [31]:
# Guardar cosas en CAche para calcuar metricas en gold y eivatr que se vaya lejos y este cerca
df_silver_enriched = df_silver_enriched.cache()
df_silver_enriched.show()

26/02/25 19:08:30 WARN CacheManager: Asked to cache already cached data.


+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+------------+-----------+-------------------+--------------+----------+--------------------+--------------+----------+--------------------+
|VendorID|          pickup_ts|         dropoff_ts|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|pickup_date|pickup_month|pickup_hour|  trip_duration_min|PU_location_id|PU_borough|             PU_zone|DO_location_id|DO_borough|             DO_zone|
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-

In [29]:
# Con todo lo que hciimos antes la data quedaron particionados en ese stage primero. Silver fue hecho con shuffle, mientras que el join ya es un broadcast

# GRoup By -> Shuffle (Columna)
# Ajustar particiones para reducir el overhead cuando tengo datasets no muy grande.
df_gold_viajes_por_zone_y_mes = (
    df_silver_enriched
    .groupby('pickup_month', 'PU_zone')
    .agg(
        F.count('*').alias('num_viajes'),
        F.avg('trip_distance').alias('avg_trip_distance')
    )
    .orderBy(F.desc('num_viajes'))
)



In [30]:
df_gold_viajes_por_zone_y_mes.show()


+------------+--------------------+----------+------------------+
|pickup_month|             PU_zone|num_viajes| avg_trip_distance|
+------------+--------------------+----------+------------------+
|     2024-01|      Midtown Center|    141829|2.5703397048558516|
|     2024-01|Upper East Side S...|    141317|1.7018326174487126|
|     2024-01|         JFK Airport|    141182|15.638710104687577|
|     2024-01|Upper East Side N...|    135407|1.8519551426440328|
|     2024-01|        Midtown East|    105503|2.2351247831815164|
|     2024-01|Times Sq/Theatre ...|    104683|2.9243368073135088|
|     2024-01|Penn Station/Madi...|    103296| 2.278204577137541|
|     2024-01| Lincoln Square East|    103038|2.0935962460451423|
|     2024-01|   LaGuardia Airport|     88579|  9.61596292574992|
|     2024-01|Upper West Side S...|     87720| 2.262874145006827|
|     2024-01|       Midtown North|     84665|  2.51198405480424|
|     2024-01|         Murray Hill|     82161| 2.198156789717759|
|     2024

In [34]:
# Ventas totale spor zona y por mes

df_gold_ventas_por_zona_y_mes = (
    df_silver_enriched
    .groupBy("pickup_month", "PU_zone")
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.count("*").alias("num_trips"),
        F.sum("fare_amount").alias("total_fare"),
        F.sum("tip_amount").alias("total_tip")
    )
    .withColumn(
        "tip_pct",
        F.col("total_tip") / F.when(F.col("total_fare") == 0, F.lit(None)).otherwise(F.col("total_fare"))
    )
    .orderBy('pickup_month', F.desc("total_revenue"))
)

df_gold_ventas_por_zona_y_mes.show()

+------------+--------------------+--------------------+---------+------------------+------------------+-------------------+
|pickup_month|             PU_zone|       total_revenue|num_trips|        total_fare|         total_tip|            tip_pct|
+------------+--------------------+--------------------+---------+------------------+------------------+-------------------+
|     2002-12|         Murray Hill|                10.5|        1|               6.5|               0.0|                0.0|
|     2009-01|   LaGuardia Airport|               68.29|        1|              50.6|               0.0|                0.0|
|     2009-01|Upper East Side S...|                50.0|        1|              45.0|               0.0|                0.0|
|     2009-01|            Kips Bay|                 9.4|        1|               4.4|               0.0|                0.0|
|     2023-12|Sutton Place/Turt...|               45.72|        1|              33.1|              7.62|0.23021148036253775|
